In [ ]:
import pandas as pd
from functools import reduce

CNY/RUB

In [ ]:
df1=pd.read_csv('/content/CBOE Volatility Index Historical Data.csv')

In [ ]:
df2=pd.read_csv('/content/CHAU ETF Stock Price History.csv')

In [ ]:
df3=pd.read_csv('/content/new_cny_rub_target_OHLC.csv')

In [ ]:
df4=pd.read_csv('/content/China 1-Year Bond Yield Historical Data.csv')

In [ ]:
df5=pd.read_csv('/content/GSCI Industrial Metals Historical Data.csv')

In [ ]:
df6=pd.read_csv('/content/MOEX Russia Index Historical Data.csv')

In [ ]:
df7=pd.read_csv('/content/US Dollar Index Historical Data.csv')

In [ ]:
df8=pd.read_csv('/content/USD_CNY.csv')

In [ ]:
df9=pd.read_csv('/content/USD_RUB.csv')

In [ ]:
df10=pd.read_csv('/content/bond_SHIBOR_3m.csv')

In [ ]:
df11=pd.read_csv('/content/china_bond_10y.csv')

In [ ]:
df12=pd.read_csv('/content/china_bond_1y.csv')

In [ ]:
df13=pd.read_csv('/content/china_bond_2y.csv')

In [ ]:
df14=pd.read_csv('/content/df_brent_oil_finaly_12api - df_brent_oil_new.csv.csv')

In [ ]:
df15=pd.read_csv('/content/russia_ofz_yeild_10year - Лист1.csv')

In [ ]:
df16=pd.read_csv('/content/russia_ofz_yeild_1year - Лист1.csv')

In [ ]:
df17=pd.read_csv('/content/russia_ofz_yeild_2year - Лист1.csv')

In [ ]:
df18=pd.read_csv('/content/russia_ofz_yeild_3m - Лист1.csv')

In [ ]:
df19=pd.read_csv('/content/true_key_rate_rf_2017_1025.csv')

In [ ]:
df20 = pd.read_csv('/content/tax_period_dummies_2017_2025 - daily_dummies.csv')

df20 = df20.rename(columns={'date': 'Date'})
df20 = df20[[
    'Date',
    'ru_tax_period_dummy',
    'ru_tax_heavy_dummy',
    'ru_tax_due_proxy',
    'cn_tax_period_dummy',
    'cn_tax_quarter_dummy',
    'cn_tax_due_proxy'
]].copy()

df21 = pd.read_csv('/content/epu_data_cn_ru - Лист1.csv')
df21 = df21.rename(columns={'date': 'Date'})

df21 = df21[[
    'Date',
    'Russia_EPU',
    'China_EPU_Factiva',
    'China_EPU_ProQuest'
]].copy()

In [ ]:
df21

,Date,Russia_EPU,China_EPU_Factiva,China_EPU_ProQuest
0,01.01.2017,400,428,695.0
1,02.01.2017,400,428,695.0
2,03.01.2017,400,428,695.0
3,04.01.2017,400,428,695.0
4,05.01.2017,400,428,695.0
...,...,...,...,...
3282,27.12.2025,433,593,NaN
3283,28.12.2025,433,593,NaN
3284,29.12.2025,433,593,NaN
3285,30.12.2025,433,593,NaN


In [ ]:
import numpy as np
import pandas as pd

def normalize_date_col(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip().str.replace("\xa0", "", regex=False)

    dt = pd.to_datetime(s, format="%d.%m.%Y", errors="coerce")
    dt = dt.fillna(pd.to_datetime(s, format="%m/%d/%Y", errors="coerce"))
    dt = dt.fillna(pd.to_datetime(s, format="%d/%m/%Y", errors="coerce"))
    dt = dt.fillna(pd.to_datetime(s, format="%Y-%m-%d", errors="coerce"))

    return dt.dt.strftime("%d.%m.%Y")

def parse_num(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace("\xa0", "").replace(" ", "").strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    if "." in s and "," in s:
        if s.rfind(",") > s.rfind("."):
            s = s.replace(".", "").replace(",", ".")
        else:
            s = s.replace(",", "")
    elif "," in s and "." not in s:
        s = s.replace(",", ".")

    try:
        return float(s)
    except:
        return np.nan


def parse_volume(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace("\xa0", "").replace(" ", "").strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    mult = 1.0
    if s[-1] in ("K", "M", "B"):
        mult = {"K": 1e3, "M": 1e6, "B": 1e9}[s[-1]]
        s = s[:-1]

    v = parse_num(s)
    return v * mult if not np.isnan(v) else np.nan


def parse_percent(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace("\xa0", "").replace(" ", "").strip()
    if s == "" or s.lower() == "nan":
        return np.nan
    s = s.replace("%", "")
    return parse_num(s)


def normalize_numeric_cells(df: pd.DataFrame, date_col="Date") -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if c == date_col:
            continue
        cl = c.lower()
        if "change" in cl and "%" in cl:
            out[c] = out[c].apply(parse_percent).astype("float64")
        elif cl in ("vol.", "volume", "vol"):
            out[c] = out[c].apply(parse_volume).astype("float64")
        else:
            out[c] = out[c].apply(parse_num).astype("float64")
    return out

def prep_table(df: pd.DataFrame, idx: int, key_col="Date") -> pd.DataFrame:
    if key_col not in df.columns:
        raise KeyError(f"В таблице с id={idx} нет столбца '{key_col}'.")

    out = df.copy()
    out[key_col] = normalize_date_col(out[key_col])
    out = out.dropna(subset=[key_col])

    out = normalize_numeric_cells(out, date_col=key_col)
    out = out.drop_duplicates(subset=[key_col], keep="last")
    mapping = {c: f"{c}_{idx}" for c in out.columns if c != key_col}
    out = out.rename(columns=mapping)
    return out

def asof_join_one(result: pd.DataFrame,
                  feat_prepped: pd.DataFrame,
                  key_col="Date",
                  direction="backward") -> pd.DataFrame:
    base = result.copy()
    feat = feat_prepped.copy()

    base["_dt"] = pd.to_datetime(base[key_col], format="%d.%m.%Y", errors="coerce")
    feat["_dt"] = pd.to_datetime(feat[key_col], format="%d.%m.%Y", errors="coerce")

    base = base.dropna(subset=["_dt"]).sort_values("_dt").reset_index(drop=True)
    feat = feat.dropna(subset=["_dt"]).sort_values("_dt").drop_duplicates(subset=["_dt"], keep="last").reset_index(drop=True)

    feat_cols = [c for c in feat.columns if c not in [key_col, "_dt"]]

    out = pd.merge_asof(
        base,
        feat[["_dt"] + feat_cols],
        on="_dt",
        direction=direction
    )

    out[key_col] = out["_dt"].dt.strftime("%d.%m.%Y")
    return out.drop(columns=["_dt"])

def build_result_asof(tables_with_ids,
                      master_id: int,
                      key_col="Date",
                      direction="backward") -> pd.DataFrame:

    prepped = {idx: prep_table(df, idx, key_col=key_col) for df, idx in tables_with_ids}

    if master_id not in prepped:
        raise ValueError(f"master_id={master_id} отсутствует в tables_with_ids")
    result = prepped[master_id][[key_col]].copy()
    for idx, feat in prepped.items():
        if idx == master_id:
            result = result.merge(feat, on=key_col, how="left")
        else:
            result = asof_join_one(result, feat, key_col=key_col, direction=direction)

    return result


tables_with_ids = [
    (df1, 1),
    (df2, 2),
    (df3, 3),
    (df4, 4),
    (df5, 5),
    (df6, 6),
    (df7, 7),
    (df8, 8),
    (df9, 9),
    (df10, 10),
    (df11, 11),
    (df12, 12),
    (df13, 13),
    (df14, 14),
    (df15, 15),
    (df16, 16),
    (df17, 17),
    (df18, 18),
    (df19, 19),
    (df20, 20),
    (df21, 21),
]

result = build_result_asof(tables_with_ids, master_id=3, key_col="Date", direction="backward")

In [ ]:
norm = normalize_date_col(result["Date"])
bad = norm.isna().sum()
print("Unparsed dates:", bad)

Unparsed dates: 0


In [ ]:
result

,Date,Price_1,Open_1,High_1,Low_1,Vol._1,Change %_1,Price_2,Open_2,High_2,...,key_rate_rf_19,ru_tax_period_dummy_20,ru_tax_heavy_dummy_20,ru_tax_due_proxy_20,cn_tax_period_dummy_20,cn_tax_quarter_dummy_20,cn_tax_due_proxy_20,Russia_EPU_21,China_EPU_Factiva_21,China_EPU_ProQuest_21
0,01.01.2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,1.0,1.0,0.0,400.0,428.0,695.0
1,02.01.2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,1.0,1.0,0.0,400.0,428.0,695.0
2,03.01.2017,12.85,14.07,14.07,12.85,NaN,-8.48,17.59,17.51,17.61,...,NaN,0.0,0.0,0.0,1.0,1.0,0.0,400.0,428.0,695.0
3,04.01.2017,11.85,12.78,12.80,11.63,NaN,-7.78,18.46,18.07,18.46,...,NaN,0.0,0.0,0.0,1.0,1.0,0.0,400.0,428.0,695.0
4,05.01.2017,11.67,11.96,12.09,11.40,NaN,-1.52,18.72,18.55,18.77,...,NaN,0.0,0.0,0.0,1.0,1.0,0.0,400.0,428.0,695.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2718,25.12.2025,13.47,14.09,14.16,13.38,NaN,-3.79,20.97,20.94,21.00,...,16.0,1.0,0.0,0.0,0.0,0.0,0.0,433.0,593.0,NaN
2719,26.12.2025,13.60,14.12,14.29,13.52,NaN,0.97,21.34,21.11,21.36,...,16.0,1.0,0.0,0.0,0.0,0.0,0.0,433.0,593.0,NaN
2720,27.12.2025,13.60,14.12,14.29,13.52,NaN,0.97,21.34,21.11,21.36,...,16.0,1.0,0.0,0.0,0.0,0.0,0.0,433.0,593.0,NaN
2721,29.12.2025,14.20,14.69,15.08,13.99,NaN,4.41,20.90,20.90,20.98,...,16.0,0.0,0.0,0.0,0.0,0.0,0.0,433.0,593.0,NaN


In [ ]:
import pandas as pd

def drop_weekends(df, date_col="Date"):
    out = df.copy()
    dt = pd.to_datetime(out[date_col], dayfirst=True, format="%d.%m.%Y", errors="coerce")
    out = out.assign(_dt=dt).dropna(subset=["_dt"])
    out = out[out["_dt"].dt.dayofweek < 5]
    out = out.sort_values("_dt").drop(columns=["_dt"]).reset_index(drop=True)
    return out
result = drop_weekends(result, "Date")

base_open_col = next((c for c in ["Open_3", "open_3"] if c in result.columns), None)
if base_open_col is not None:
    result = result.dropna(subset=[base_open_col])


In [ ]:
result = result.drop(columns=['Vol._1', 'Vol._5', 'Vol._6', 'Vol._8', 'Vol._9'], errors='ignore')


In [ ]:
import pandas as pd

def nan_report(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    rep = pd.DataFrame({
        "nan_count": df.isna().sum(),
        "nan_pct": (df.isna().mean() * 100).round(2),
        "dtype": df.dtypes.astype(str),
        "non_nan_count": n - df.isna().sum()
    })
    rep = rep.sort_values(["nan_pct", "nan_count"], ascending=False)
    return rep

report = nan_report(result)
display(report)


,nan_count,nan_pct,dtype,non_nan_count
Vol._7,2347,100.00,float64,0
China_EPU_ProQuest_21,543,23.14,float64,1804
key_rate_rf_19,5,0.21,float64,2342
Vol._2,3,0.13,float64,2344
Price_1,1,0.04,float64,2346
...,...,...,...,...
cn_tax_period_dummy_20,0,0.00,float64,2347
cn_tax_quarter_dummy_20,0,0.00,float64,2347
cn_tax_due_proxy_20,0,0.00,float64,2347
Russia_EPU_21,0,0.00,float64,2347


In [ ]:
result

,Date,Price_1,Open_1,High_1,Low_1,Change %_1,Price_2,Open_2,High_2,Low_2,...,key_rate_rf_19,ru_tax_period_dummy_20,ru_tax_heavy_dummy_20,ru_tax_due_proxy_20,cn_tax_period_dummy_20,cn_tax_quarter_dummy_20,cn_tax_due_proxy_20,Russia_EPU_21,China_EPU_Factiva_21,China_EPU_ProQuest_21
0,02.01.2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,1.0,1.0,0.0,400.0,428.0,695.0
1,03.01.2017,12.85,14.07,14.07,12.85,-8.48,17.59,17.51,17.61,17.50,...,NaN,0.0,0.0,0.0,1.0,1.0,0.0,400.0,428.0,695.0
2,04.01.2017,11.85,12.78,12.80,11.63,-7.78,18.46,18.07,18.46,18.07,...,NaN,0.0,0.0,0.0,1.0,1.0,0.0,400.0,428.0,695.0
3,05.01.2017,11.67,11.96,12.09,11.40,-1.52,18.72,18.55,18.77,18.55,...,NaN,0.0,0.0,0.0,1.0,1.0,0.0,400.0,428.0,695.0
4,06.01.2017,11.32,11.70,11.74,10.98,-3.00,18.11,18.38,18.38,18.05,...,NaN,0.0,0.0,0.0,1.0,1.0,0.0,400.0,428.0,695.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2342,24.12.2025,13.47,14.09,14.16,13.38,-3.79,20.97,20.94,21.00,20.85,...,16.0,1.0,0.0,0.0,0.0,0.0,0.0,433.0,593.0,NaN
2343,25.12.2025,13.47,14.09,14.16,13.38,-3.79,20.97,20.94,21.00,20.85,...,16.0,1.0,0.0,0.0,0.0,0.0,0.0,433.0,593.0,NaN
2344,26.12.2025,13.60,14.12,14.29,13.52,0.97,21.34,21.11,21.36,21.11,...,16.0,1.0,0.0,0.0,0.0,0.0,0.0,433.0,593.0,NaN
2345,29.12.2025,14.20,14.69,15.08,13.99,4.41,20.90,20.90,20.98,20.86,...,16.0,0.0,0.0,0.0,0.0,0.0,0.0,433.0,593.0,NaN


In [ ]:
import numpy as np
import pandas as pd

def add_ohlcv_features(df: pd.DataFrame, suffix: str, prefix: str, make_d_bp: bool = False) -> pd.DataFrame:
    out = df.copy()
    close_cand = [f"Price_{suffix}", f"Close_{suffix}", f"close_{suffix}", f"price_{suffix}"]
    open_cand  = [f"Open_{suffix}",  f"open_{suffix}"]
    high_cand  = [f"High_{suffix}",  f"high_{suffix}"]
    low_cand   = [f"Low_{suffix}",   f"low_{suffix}"]
    vol_cand   = [f"Vol._{suffix}", f"vol._{suffix}", f"Volume_{suffix}", f"volume_{suffix}"]

    def pick(cols):
        for c in cols:
            if c in out.columns:
                return c
        return None

    P = pick(close_cand)
    O = pick(open_cand)
    H = pick(high_cand)
    L = pick(low_cand)
    V = pick(vol_cand)
    if P is None:
        return out
    for c in [P, O, H, L, V]:
        if c is not None:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    lvl = out[P].ffill()
    out[f"{prefix}_level"] = lvl
    if make_d_bp:
        out[f"{prefix}_d_bp"] = (lvl.diff() * 100.0).fillna(0.0)
    loglvl = np.log(lvl.where(lvl > 0))
    out[f"{prefix}_logret"] = loglvl.diff()
    if H is not None and L is not None:
        hi = out[H].ffill()
        lo = out[L].ffill()
        out[f"{prefix}_range"] = np.log((hi / lo).where((hi > 0) & (lo > 0)))

    if O is not None:
        op = out[O].ffill()
        out[f"{prefix}_oc"] = np.log((lvl / op).where((lvl > 0) & (op > 0)))
    if V is not None and out[V].notna().any():
        out[f"{prefix}_volume"] = out[V].astype("float64")
        vol = out[f"{prefix}_volume"].ffill()
        out[f"{prefix}_logvol"] = np.log(vol.where(vol > 0))
        out[f"{prefix}_dlogvol"] = out[f"{prefix}_logvol"].diff()

    return out

In [ ]:
result = add_ohlcv_features(result, suffix="1",  prefix="vix",   make_d_bp=False)


In [ ]:
result = add_ohlcv_features(result, suffix="2",  prefix="chau",   make_d_bp=False)


In [ ]:
result = add_ohlcv_features(result, suffix="4",  prefix="china_1y_bond_yield",   make_d_bp=True)

In [ ]:
result = add_ohlcv_features(result, suffix="5",  prefix="gsci_metal",   make_d_bp=False)

In [ ]:
result = add_ohlcv_features(result, suffix="6",  prefix="moex",   make_d_bp=False)

In [ ]:
result = add_ohlcv_features(result, suffix="7",  prefix="usd_index",   make_d_bp=False)

In [ ]:
result = add_ohlcv_features(result, suffix="8",  prefix="usd_cny",   make_d_bp=False)

In [ ]:
result = add_ohlcv_features(result, suffix="9",  prefix="usd_rub",   make_d_bp=False)

In [ ]:
result = add_ohlcv_features(result, suffix="10",  prefix="shibor",   make_d_bp=True)

In [ ]:
result = add_ohlcv_features(result, suffix="11",  prefix="china_10y_bond",   make_d_bp=True)

In [ ]:
result = add_ohlcv_features(result, suffix="12",  prefix="china_1y_bond",   make_d_bp=True)

In [ ]:
result = add_ohlcv_features(result, suffix="13",  prefix="china_2y_bond",   make_d_bp=True)

In [ ]:
result = add_ohlcv_features(result, suffix="14",  prefix="brent",   make_d_bp=False)

In [ ]:
result = add_ohlcv_features(result, suffix="6",  prefix="moex",   make_d_bp=False)

In [ ]:
result['chau_logret']

,chau_logret
0,NaN
1,NaN
2,0.048276
3,0.013986
4,-0.033128
...,...
2342,0.004301
2343,0.000000
2344,0.017490
2345,-0.020834


In [ ]:
import numpy as np
import pandas as pd

def add_yield_features(df: pd.DataFrame, y_col: str, prefix: str, use_ffill: bool = True) -> pd.DataFrame:

    out = df.copy()

    if y_col not in out.columns:
        out[f"{prefix}_level"] = np.nan
        out[f"{prefix}_d_bp"] = np.nan
        return out

    out[y_col] = pd.to_numeric(out[y_col], errors="coerce").astype("float64")

    lvl = out[y_col].ffill() if use_ffill else out[y_col]
    out[f"{prefix}_level"] = lvl
    out[f"{prefix}_d_bp"] = (lvl.diff() * 100.0).fillna(0.0)

    return out


def prepare_russian_bonds(result: pd.DataFrame) -> pd.DataFrame:
    df = result.copy()

    df = add_yield_features(df, "ofz_yeild_10year_15", "ru10y", use_ffill=True)
    df = add_yield_features(df, "ofz_yeild_1year_16",  "ru1y",  use_ffill=True)
    df = add_yield_features(df, "ofz_yeild_2year_17",  "ru2y",  use_ffill=True)
    df = add_yield_features(df, "ofz_yeild_3m_18",     "ru3m",  use_ffill=True)
    if "ru10y_level" in df.columns and "ru2y_level" in df.columns:
        df["ru_spread_10y_2y"] = df["ru10y_level"] - df["ru2y_level"]
    if "ru2y_level" in df.columns and "ru1y_level" in df.columns:
        df["ru_spread_2y_1y"] = df["ru2y_level"] - df["ru1y_level"]
    if "ru1y_level" in df.columns and "ru3m_level" in df.columns:
        df["ru_spread_1y_3m"] = df["ru1y_level"] - df["ru3m_level"]

    return df

In [ ]:
import numpy as np
import pandas as pd

def add_yield_features_full(df: pd.DataFrame,
                            y_col: str,
                            prefix: str,
                            use_ffill: bool = True,
                            eps: float = 1e-6) -> pd.DataFrame:
    out = df.copy()

    if y_col not in out.columns:
        return out
    out[y_col] = pd.to_numeric(out[y_col], errors="coerce").astype("float64")

    lvl = out[y_col].ffill() if use_ffill else out[y_col]
    out[f"{prefix}_level"] = lvl
    dbp = (lvl.diff() * 100.0).fillna(0.0)
    out[f"{prefix}_d_bp"] = dbp
    safe = lvl.clip(lower=eps)
    out[f"{prefix}_logret"] = np.log(safe).diff()
    out[f"{prefix}_d_abs_bp"] = dbp.abs()
    out[f"{prefix}_dbp_ma_5"] = dbp.rolling(5).mean()
    out[f"{prefix}_dbp_std_20"] = dbp.rolling(20).std()

    return out


def prepare_russian_bonds_full(result: pd.DataFrame) -> pd.DataFrame:
    df = result.copy()

    df = add_yield_features_full(df, "ofz_yeild_10year_15", "ru10y")
    df = add_yield_features_full(df, "ofz_yeild_2year_17",  "ru2y")
    df = add_yield_features_full(df, "ofz_yeild_1year_16",  "ru1y")
    df = add_yield_features_full(df, "ofz_yeild_3m_18",     "ru3m")
    if {"ru10y_level", "ru2y_level"}.issubset(df.columns):
        df["ru_spread_10y_2y"] = df["ru10y_level"] - df["ru2y_level"]
    if {"ru2y_level", "ru1y_level"}.issubset(df.columns):
        df["ru_spread_2y_1y"] = df["ru2y_level"] - df["ru1y_level"]
    if {"ru1y_level", "ru3m_level"}.issubset(df.columns):
        df["ru_spread_1y_3m"] = df["ru1y_level"] - df["ru3m_level"]
    if {"ru10y_level", "ru3m_level"}.issubset(df.columns):
        df["ru_spread_10y_3m"] = df["ru10y_level"] - df["ru3m_level"]
    for s in ["ru_spread_10y_2y", "ru_spread_2y_1y", "ru_spread_1y_3m", "ru_spread_10y_3m"]:
        if s in df.columns:
            df[f"{s}_d_bp"] = (df[s].ffill().diff() * 100.0).fillna(0.0)

    return df

In [ ]:
result=prepare_russian_bonds_full(result)

In [ ]:
import numpy as np
import pandas as pd

def prepare_key_rate_rf(df: pd.DataFrame, suffix: str = "19", date_col: str = "Date") -> pd.DataFrame:
    out = df.copy()
    candidates = [
        f"key_rate_rf_level_{suffix}",
        f"key_rate_rf_level_raw_{suffix}",
        f"key_rate_rf_{suffix}",
        f"key_rate_{suffix}",
        f"key_rate_rf_level",
        f"key_rate_rf_level_raw",
        "key_rate_rf_level",
        "key_rate_rf_level_raw",
        "key_rate_rf",
        "key_rate",
    ]
    rate_col = next((c for c in candidates if c in out.columns), None)
    if rate_col is None:
        return out
    out["key_rate_rf_level"] = out[rate_col].ffill()
    out["key_rate_rf_d_bp"] = out["key_rate_rf_level"].diff() * 100.0
    out["key_rate_rf_d_bp"] = out["key_rate_rf_d_bp"].fillna(0.0)

    out["key_rate_rf_changed"] = (out["key_rate_rf_d_bp"] != 0).astype("int8")

    last_change_dt = out["key_rate_rf_changed"].replace(0, np.nan)
    last_change_idx = np.where(out["key_rate_rf_changed"].to_numpy() == 1, np.arange(len(out)), np.nan)
    last_change_idx = pd.Series(last_change_idx).ffill()
    out["key_rate_rf_days_since_change"] = (pd.Series(np.arange(len(out))) - last_change_idx).astype("float64")
    out.loc[last_change_idx.isna(), "key_rate_rf_days_since_change"] = np.nan

    return out


In [ ]:
result = prepare_key_rate_rf(result, suffix="19", date_col="Date")


In [ ]:
result

,Date,Price_1,Open_1,High_1,Low_1,Change %_1,Price_2,Open_2,High_2,Low_2,...,ru_spread_1y_3m,ru_spread_10y_3m,ru_spread_10y_2y_d_bp,ru_spread_2y_1y_d_bp,ru_spread_1y_3m_d_bp,ru_spread_10y_3m_d_bp,key_rate_rf_level,key_rate_rf_d_bp,key_rate_rf_changed,key_rate_rf_days_since_change
0,02.01.2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,NaN,0.0,0,NaN
1,03.01.2017,12.85,14.07,14.07,12.85,-8.48,17.59,17.51,17.61,17.50,...,-0.06,-0.04,0.0,0.0,0.0,0.0,NaN,0.0,0,NaN
2,04.01.2017,11.85,12.78,12.80,11.63,-7.78,18.46,18.07,18.46,18.07,...,-0.02,0.03,9.0,-6.0,4.0,7.0,NaN,0.0,0,NaN
3,05.01.2017,11.67,11.96,12.09,11.40,-1.52,18.72,18.55,18.77,18.55,...,-0.13,-0.13,-6.0,1.0,-11.0,-16.0,NaN,0.0,0,NaN
4,06.01.2017,11.32,11.70,11.74,10.98,-3.00,18.11,18.38,18.38,18.05,...,0.01,0.01,-9.0,9.0,14.0,14.0,NaN,0.0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2342,24.12.2025,13.47,14.09,14.16,13.38,-3.79,20.97,20.94,21.00,20.85,...,0.89,1.51,11.0,5.0,7.0,23.0,16.0,0.0,0,2.0
2343,25.12.2025,13.47,14.09,14.16,13.38,-3.79,20.97,20.94,21.00,20.85,...,0.94,1.61,4.0,1.0,5.0,10.0,16.0,0.0,0,3.0
2344,26.12.2025,13.60,14.12,14.29,13.52,0.97,21.34,21.11,21.36,21.11,...,0.94,1.72,4.0,7.0,0.0,11.0,16.0,0.0,0,4.0
2345,29.12.2025,14.20,14.69,15.08,13.99,4.41,20.90,20.90,20.98,20.86,...,0.88,1.72,9.0,-3.0,-6.0,0.0,16.0,0.0,0,5.0


In [ ]:
import numpy as np
import pandas as pd

def prepare_tax_dummies(df: pd.DataFrame, suffix: str = "20") -> pd.DataFrame:
    out = df.copy()

    tax_map = {
        f"ru_tax_period_dummy_{suffix}": "ru_tax_period_dummy",
        f"ru_tax_heavy_dummy_{suffix}": "ru_tax_heavy_dummy",
        f"ru_tax_due_proxy_{suffix}": "ru_tax_due_proxy",
        f"cn_tax_period_dummy_{suffix}": "cn_tax_period_dummy",
        f"cn_tax_quarter_dummy_{suffix}": "cn_tax_quarter_dummy",
        f"cn_tax_due_proxy_{suffix}": "cn_tax_due_proxy",
    }

    for raw_col, final_col in tax_map.items():
        if raw_col in out.columns:
            out[final_col] = pd.to_numeric(out[raw_col], errors="coerce").fillna(0).astype("int8")

    if {"ru_tax_period_dummy", "cn_tax_period_dummy"}.issubset(out.columns):
        out["tax_overlap_ru_cn"] = (
            (out["ru_tax_period_dummy"] == 1) &
            (out["cn_tax_period_dummy"] == 1)
        ).astype("int8")

    if {"ru_tax_due_proxy", "cn_tax_due_proxy"}.issubset(out.columns):
        out["tax_due_any"] = (
            (out["ru_tax_due_proxy"] == 1) |
            (out["cn_tax_due_proxy"] == 1)
        ).astype("int8")

    return out


def add_epu_features(df: pd.DataFrame, suffix: str = "21", eps: float = 1e-6) -> pd.DataFrame:
    out = df.copy()

    epu_map = {
        f"Russia_EPU_{suffix}": "epu_ru",
        f"China_EPU_Factiva_{suffix}": "epu_cn_factiva",
        f"China_EPU_ProQuest_{suffix}": "epu_cn_proquest",
    }

    for raw_col, prefix in epu_map.items():
        if raw_col in out.columns:
            lvl = pd.to_numeric(out[raw_col], errors="coerce").ffill()
            out[f"{prefix}_level"] = lvl
            out[f"{prefix}_logret"] = np.log(lvl.clip(lower=eps)).diff()
            out[f"{prefix}_d_pct"] = lvl.pct_change()
    cn_cols = [c for c in ["epu_cn_factiva_level", "epu_cn_proquest_level"] if c in out.columns]
    if len(cn_cols) > 0:
        out["epu_cn_avg_level"] = out[cn_cols].mean(axis=1)
        out["epu_cn_avg_logret"] = np.log(out["epu_cn_avg_level"].clip(lower=eps)).diff()
    if {"epu_ru_level", "epu_cn_avg_level"}.issubset(out.columns):
        out["epu_diff_ru_cn_level"] = out["epu_ru_level"] - out["epu_cn_avg_level"]
        out["epu_diff_ru_cn_log"] = (
            np.log(out["epu_ru_level"].clip(lower=eps)) -
            np.log(out["epu_cn_avg_level"].clip(lower=eps))
        )

    return out


result = prepare_tax_dummies(result, suffix="20")
result = add_epu_features(result, suffix="21")

In [ ]:
result = prepare_tax_dummies(result)

In [ ]:
result = add_epu_features(result)

In [ ]:
result['tax_overlap_ru_cn'][result['tax_overlap_ru_cn']==1]

,tax_overlap_ru_cn


In [ ]:
raw_drop = [
    "ru_tax_period_dummy_20",
    "ru_tax_heavy_dummy_20",
    "ru_tax_due_proxy_20",
    "cn_tax_period_dummy_20",
    "cn_tax_quarter_dummy_20",
    "cn_tax_due_proxy_20",
    "Russia_EPU_21",
    "China_EPU_Factiva_21",
    "China_EPU_ProQuest_21",
    # "ru_tax_period_dummy",
    # "cn_tax_period_dummy",
    "ru_tax_heavy_dummy",
    "ru_tax_due_proxy",
    "cn_tax_quarter_dummy",
    "cn_tax_due_proxy",
    "tax_due_any",
    "epu_ru_logret",
    "epu_ru_d_pct",
    "epu_cn_factiva_logret",
    "epu_cn_factiva_d_pct",
    "epu_cn_proquest_logret",
    "epu_cn_proquest_d_pct",
    "epu_cn_avg_logret",
    "tax_overlap_ru_cn"


]

result = result.drop(columns=[c for c in raw_drop if c in result.columns], errors="ignore")

In [ ]:
import pandas as pd

def nan_report(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    rep = pd.DataFrame({
        "nan_count": df.isna().sum(),
        "nan_pct": (df.isna().mean() * 100).round(2),
        "dtype": df.dtypes.astype(str),
        "non_nan_count": n - df.isna().sum()
    })
    rep = rep.sort_values(["nan_pct", "nan_count"], ascending=False)
    return rep

report = nan_report(result)
display(report)


,nan_count,nan_pct,dtype,non_nan_count
Vol._7,2347,100.00,float64,0
key_rate_rf_days_since_change,60,2.56,float64,2287
ru10y_dbp_std_20,19,0.81,float64,2328
ru2y_dbp_std_20,19,0.81,float64,2328
ru1y_dbp_std_20,19,0.81,float64,2328
...,...,...,...,...
epu_cn_factiva_level,0,0.00,float64,2347
epu_cn_proquest_level,0,0.00,float64,2347
epu_cn_avg_level,0,0.00,float64,2347
epu_diff_ru_cn_level,0,0.00,float64,2347


In [ ]:
import numpy as np
import pandas as pd
def make_vol_targets_ohlc(df: pd.DataFrame, suffix: str = "3", date_col: str = "Date") -> pd.DataFrame:
    out = df.copy()

    C = f"Price_{suffix}"
    O = f"Open_{suffix}"
    H = f"High_{suffix}"
    L = f"Low_{suffix}"

    if C not in out.columns and f"Close_{suffix}" in out.columns:
        C = f"Close_{suffix}"
    if O not in out.columns and f"open_{suffix}" in out.columns:
        O = f"open_{suffix}"
    if H not in out.columns and f"high_{suffix}" in out.columns:
        H = f"high_{suffix}"
    if L not in out.columns and f"low_{suffix}" in out.columns:
        L = f"low_{suffix}"

    if any(c not in out.columns for c in [C, O, H, L]):
        return out
    out["cnyrub_close"] = out[C]
    out["cnyrub_log_close"] = np.log(out["cnyrub_close"].where(out["cnyrub_close"] > 0))
    out["cnyrub_logret"] = out["cnyrub_log_close"].diff()

    hl = np.log((out[H] / out[L]).where((out[H] > 0) & (out[L] > 0)))
    out["cnyrub_range"] = hl
    out["cnyrub_oc"] = np.log((out[C] / out[O]).where((out[C] > 0) & (out[O] > 0)))
    out["cnyrub_gap"] = np.log((out[O] / out[C].shift(1)).where((out[O] > 0) & (out[C].shift(1) > 0)))

    co = out["cnyrub_oc"]
    out["RV_gk"] = 0.5*(hl**2) - (2*np.log(2)-1)*(co**2)
    out["target_RV_gk"] = out["RV_gk"].shift(-1)
    out["target_vol_gk"] = np.sqrt(out["target_RV_gk"])

    return out


In [ ]:
result=make_vol_targets_ohlc(result)

In [ ]:
result.head()

,Date,Price_1,Open_1,High_1,Low_1,Change %_1,Price_2,Open_2,High_2,Low_2,...,epu_diff_ru_cn_log,cnyrub_close,cnyrub_log_close,cnyrub_logret,cnyrub_range,cnyrub_oc,cnyrub_gap,RV_gk,target_RV_gk,target_vol_gk
0,02.01.2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.339147,8.82197,2.177245,NaN,0.000116,-0.000101,NaN,2.752085e-09,0.000120,0.010957
1,03.01.2017,12.85,14.07,14.07,12.85,-8.48,17.59,17.51,17.61,17.50,...,-0.339147,8.75172,2.169250,-0.007995,0.017041,-0.008067,0.000073,1.200513e-04,0.000083,0.009136
2,04.01.2017,11.85,12.78,12.80,11.63,-7.78,18.46,18.07,18.46,18.07,...,-0.339147,8.70746,2.164180,-0.005070,0.013641,-0.004978,-0.000093,8.346545e-05,0.000214,0.014624
3,05.01.2017,11.67,11.96,12.09,11.40,-1.52,18.72,18.55,18.77,18.55,...,-0.339147,8.63810,2.156183,-0.007997,0.022235,-0.009291,0.001293,2.138501e-04,0.000051,0.007129
4,06.01.2017,11.32,11.70,11.74,10.98,-3.00,18.11,18.38,18.38,18.05,...,-0.339147,8.61638,2.153665,-0.002518,0.010171,-0.001527,-0.000990,5.082004e-05,0.000028,0.005252


In [ ]:
cols_drop = [c for c in ["Price_3", "Open_3", "High_3", "Low_3", "Change %_3"] if c in result.columns]
result = result.drop(columns=cols_drop)


In [ ]:
if {"cnyrub_close", "usdrub_close", "usdcny_close"}.issubset(result.columns):
    result["cnyrub_implied"] = result["usdrub_close"] / result["usdcny_close"]
    result["cnyrub_basis"] = np.log(result["cnyrub_close"]) - np.log(result["cnyrub_implied"])
    result["cnyrub_basis_abs"] = result["cnyrub_basis"].abs()
    result["cnyrub_basis_std_20"] = result["cnyrub_basis"].rolling(20).std()


In [ ]:
import pandas as pd

def nan_report(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    rep = pd.DataFrame({
        "nan_count": df.isna().sum(),
        "nan_pct": (df.isna().mean() * 100).round(2),
        "dtype": df.dtypes.astype(str),
        "non_nan_count": n - df.isna().sum()
    })
    rep = rep.sort_values(["nan_pct", "nan_count"], ascending=False)
    return rep
report = nan_report(result)
display(report)

,nan_count,nan_pct,dtype,non_nan_count
Vol._7,2347,100.00,float64,0
key_rate_rf_days_since_change,60,2.56,float64,2287
ru10y_dbp_std_20,19,0.81,float64,2328
ru2y_dbp_std_20,19,0.81,float64,2328
ru1y_dbp_std_20,19,0.81,float64,2328
...,...,...,...,...
cnyrub_close,0,0.00,float64,2347
cnyrub_log_close,0,0.00,float64,2347
cnyrub_range,0,0.00,float64,2347
cnyrub_oc,0,0.00,float64,2347


In [ ]:
result=result.drop(columns=['Vol._7'])

In [ ]:
path = "result_hope_8_march.xlsx"
result.to_excel(path, index=False)